# Stage 6: Held-out test evaluation

Stage 6 selects one final specification for MLR, RF, XGBoost and supplementary BRF using the 7,619-participant training sample, fits each model on all training participants, and evaluates the models once on the 1,905-participant held-out test sample.

Macro F1 is the selection criterion. Before held-out evaluation, test identifiers are used only to verify separation from the training sample; test-set predictors and outcomes are not used for preprocessing, weighting or model selection.


## Part 1: Inputs and checks

In [1]:
# 1: Imports and project paths

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

from imblearn.ensemble import BalancedRandomForestClassifier
from xgboost import XGBClassifier

working_directory = Path.cwd().resolve()

project_root = next(
    (
        directory
        for directory in [working_directory, *working_directory.parents]
        if (
            directory
            / "data_derived"
            / "stage_3_final_modelling_dataset"
            / "final_modelling_dataset.csv"
        ).is_file()
        and (
            directory
            / "data_derived"
            / "stage_5_pre_test_readiness_audit"
            / "stage_5_pre_test_completion.csv"
        ).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "The Stage 3 modelling dataset and Stage 5.2 readiness file could not be located."
    )

data_derived = project_root / "data_derived"

stage_3_dir = data_derived / "stage_3_final_modelling_dataset"
stage_4_dir = data_derived / "stage_4_modelling_design"
stage_5_dir = data_derived / "stage_5_nested_model_development"
audit_dir = data_derived / "stage_5_pre_test_readiness_audit"
stage_6_dir = data_derived / "stage_6_held_out_test_evaluation"
stage_6_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "data": stage_3_dir / "final_modelling_dataset.csv",
    "split": stage_4_dir / "stage_4_train_test_split.csv",
    "folds": stage_4_dir / "stage_4_outer_folds.csv",
    "roles": stage_4_dir / "stage_4_predictor_preprocessing_roles.csv",
    "search": stage_4_dir / "stage_4_model_search_specification.csv",
    "candidates": stage_5_dir / "stage_5_hyperparameter_candidates.csv",
    "stage5_audit": stage_5_dir / "stage_5_final_audit.csv",
    "readiness": audit_dir / "stage_5_pre_test_completion.csv",
}

missing = [name for name, path in paths.items() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing required input file(s): " + ", ".join(missing)
    )

print("Project root located.")
print(
    "Stage 6 output directory: "
    f"{stage_6_dir.relative_to(project_root).as_posix()}"
)


Project root located.
Stage 6 output directory: data_derived/stage_6_held_out_test_evaluation


In [2]:
# 2: Load the training sample and design files

split = pd.read_csv(paths["split"], dtype={"NSID": "string"})
folds = pd.read_csv(paths["folds"], dtype={"NSID": "string"})
roles = pd.read_csv(paths["roles"])
search_spec = pd.read_csv(paths["search"])
candidates = pd.read_csv(
    paths["candidates"],
    keep_default_na=False,
    na_values=[""],
)
stage5_audit = pd.read_csv(paths["stage5_audit"])
readiness = pd.read_csv(paths["readiness"])

train_ids = set(
    split.loc[split["sample"].eq("Training"), "NSID"]
)
test_ids = set(
    split.loc[split["sample"].eq("Test"), "NSID"]
)

data = pd.read_csv(paths["data"], dtype={"NSID": "string"})
train = (
    data.loc[data["NSID"].isin(train_ids)]
    .copy()
    .reset_index(drop=True)
)
del data

predictors = roles["Predictor"].tolist()

print(f"Training participants: {len(train):,}")
print(f"Test identifiers: {len(test_ids):,}")
print(f"Predictors: {len(predictors)}")
print(f"Candidate configurations: {len(candidates)}")


Training participants: 7,619
Test identifiers: 1,905
Predictors: 69
Candidate configurations: 189


In [3]:
# 3: Input audit

expected_outcomes = {
    1: "Education",
    4: "Employment",
    5: "Apprenticeship or training",
    6: "Unemployment or inactivity (NEET)",
}

outcome_map = dict(
    train[["age18_outcome_code", "age18_outcome"]]
    .drop_duplicates()
    .sort_values("age18_outcome_code")
    .astype({"age18_outcome_code": int})
    .itertuples(index=False, name=None)
)

candidate_counts = candidates.groupby("Model").size().to_dict()


def to_boolean_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
        .fillna(False)
        .astype(bool)
    )


required_readiness_columns = {
    "Required checks passed",
    "Failed required checks",
}
if not required_readiness_columns.issubset(readiness.columns):
    raise ValueError(
        "The Stage 5.2 completion file does not contain the expected readiness fields."
    )

ready = readiness.iloc[0]
readiness_passed = (
    str(ready["Required checks passed"]).strip().lower() == "true"
    and int(ready["Failed required checks"]) == 0
)

checks = {
    "Training sample contains 7,619 participants": len(train) == 7_619,
    "Held-out sample contains 1,905 identifiers": len(test_ids) == 1_905,
    "Training and test identifiers do not overlap": train_ids.isdisjoint(test_ids),
    "Training identifiers are unique": train["NSID"].is_unique,
    "All 69 predictors are registered once": (
        len(roles) == 69 and roles["Predictor"].is_unique
    ),
    "All predictors are present": set(predictors).issubset(train.columns),
    "Outcome contains the four fixed classes": outcome_map == expected_outcomes,
    "Five validation folds cover the training sample": (
        set(folds["outer_fold"]) == {1, 2, 3, 4, 5}
        and set(folds["NSID"]) == train_ids
    ),
    "Candidate counts are 9, 60, 60 and 60": candidate_counts == {
        "MLR": 9,
        "RF": 60,
        "XGBoost": 60,
        "BRF": 60,
    },
    "Macro F1 is the selection metric": (
        search_spec["Selection metric"].eq("Macro F1").all()
    ),
    "Stage 5 audit passed": to_boolean_series(
        stage5_audit["Passed"]
    ).all(),
    "Pre-test readiness audit passed": readiness_passed,
}

input_audit = pd.DataFrame(
    {"Check": checks.keys(), "Passed": checks.values()}
)
display(input_audit)

if not input_audit["Passed"].all():
    failed = input_audit.loc[
        ~input_audit["Passed"], "Check"
    ].tolist()
    raise AssertionError("; ".join(failed))


,Check,Passed
0,"Training sample contains 7,619 participants",True
1,"Held-out sample contains 1,905 identifiers",True
2,Training and test identifiers do not overlap,True
3,Training identifiers are unique,True
4,All 69 predictors are registered once,True
5,All predictors are present,True
6,Outcome contains the four fixed classes,True
7,Five validation folds cover the training sample,True
8,"Candidate counts are 9, 60, 60 and 60",True
9,Macro F1 is the selection metric,True


## Part 2: Model specification

Numeric and ordinal predictors use median imputation. Binary and nominal predictors use most-frequent imputation and one-hot encoding. MLR also standardises numeric and ordinal predictors because its L2 penalty is scale-dependent; the tree models do not require scaling.

MLR, RF and XGBoost compare unweighted and balanced fitting. BRF uses internal balanced sampling. No external over-sampling or under-sampling is used.

In [4]:
# 1: Predictor roles and outcome coding

numeric = roles.loc[roles["Preprocessing role"].eq("Numeric"), "Predictor"].tolist()
categorical = roles.loc[roles["Preprocessing role"].eq("Categorical"), "Predictor"].tolist()

assert len(numeric) + len(categorical) == 69
assert set(numeric).isdisjoint(categorical)

for column in numeric:
    train[column] = pd.to_numeric(train[column], errors="raise")

code_to_model = {1: 0, 4: 1, 5: 2, 6: 3}
model_to_code = {v: k for k, v in code_to_model.items()}
model_to_label = {code_to_model[k]: v for k, v in expected_outcomes.items()}
classes = [0, 1, 2, 3]

train["y"] = train["age18_outcome_code"].map(code_to_model).astype(int)

model_order = ["MLR", "RF", "XGBoost", "BRF"]
file_stub = {"MLR": "mlr", "RF": "rf", "XGBoost": "xgboost", "BRF": "brf"}

display(
    pd.DataFrame({
        "Role": ["Numeric / ordinal", "Binary / nominal"],
        "Predictors": [len(numeric), len(categorical)]
    })
)

,Role,Predictors
0,Numeric / ordinal,38
1,Binary / nominal,31


In [5]:
# 2: Read the fixed model random state from the design register

search_by_model = search_spec.set_index("Model")

seed_models = [
    "Random forest",
    "XGBoost",
    "Balanced random forest",
]

model_random_states = {
    model_name: int(
        json.loads(
            search_by_model.loc[
                model_name,
                "Fixed parameters",
            ]
        )["random_state"]
    )
    for model_name in seed_models
}

if len(set(model_random_states.values())) != 1:
    raise ValueError(
        "Tree-model random states are not consistent across the Stage 4 design register."
    )

MODEL_RANDOM_STATE = next(iter(model_random_states.values()))

print(f"Model random state: {MODEL_RANDOM_STATE}")


Model random state: 424242


In [6]:
# 3: Preprocessing and estimator functions

def make_preprocessor(model):
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if model == "MLR":
        num_steps.append(("scaler", StandardScaler()))

    return ColumnTransformer([
        ("numeric", Pipeline(num_steps), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", drop="if_binary")),
        ]), categorical),
    ])


def make_estimator(model, params, weighting):
    if model == "MLR":
        return LogisticRegression(
            C=float(params["C"]), solver="lbfgs", max_iter=3000,
            class_weight="balanced" if weighting == "Balanced" else None
        )

    if model == "RF":
        return RandomForestClassifier(
            **params, min_samples_split=2,
            class_weight="balanced" if weighting == "Balanced" else None,
            n_jobs=-1, random_state=MODEL_RANDOM_STATE
        )

    if model == "XGBoost":
        return XGBClassifier(
            **params, objective="multi:softprob", num_class=4,
            eval_metric="mlogloss", tree_method="hist",
            n_jobs=-1, random_state=MODEL_RANDOM_STATE, verbosity=0
        )

    return BalancedRandomForestClassifier(
        **params, min_samples_split=2,
        sampling_strategy="all", replacement=True, bootstrap=False,
        class_weight=None, n_jobs=-1, random_state=MODEL_RANDOM_STATE
    )


def make_pipeline(model, params, weighting):
    return Pipeline([
        ("preprocessor", make_preprocessor(model)),
        ("model", make_estimator(model, params, weighting)),
    ])


def fit_pipeline(pipe, X, y, model, weighting):
    # Balanced XGBoost weights are calculated from the current fitting sample,
    # which avoids using class frequencies from validation observations.
    if model == "XGBoost" and weighting == "Balanced":
        weights = compute_sample_weight("balanced", y)
        pipe.fit(X, y, model__sample_weight=weights)
    else:
        pipe.fit(X, y)
    return pipe


def weighting_modes(model):
    return ["Internal balanced sampling"] if model == "BRF" else ["None", "Balanced"]


def weighting_priority(mode):
    return {"None": 0, "Balanced": 1, "Internal balanced sampling": 0}[mode]

In [7]:
# 4: Evaluation functions

def overall_metrics(y_true, y_pred):
    return {
        "Macro F1": f1_score(y_true, y_pred, labels=classes, average="macro", zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Weighted F1": f1_score(y_true, y_pred, labels=classes, average="weighted", zero_division=0),
    }


def class_metrics(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=classes, zero_division=0
    )
    return pd.DataFrame({
        "Internal class": classes,
        "Outcome code": [model_to_code[c] for c in classes],
        "Outcome": [model_to_label[c] for c in classes],
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Support": support.astype(int),
    })

In [8]:
# 5: Fixed five-fold validation indices

cv = {}
for fold in range(1, 6):
    val_ids = set(folds.loc[folds["outer_fold"].eq(fold), "NSID"])
    cv[fold] = (
        train.index[~train["NSID"].isin(val_ids)].to_numpy(),
        train.index[train["NSID"].isin(val_ids)].to_numpy(),
    )

cv_summary = pd.DataFrame([
    {
        "Fold": fold,
        "Training": len(index[0]),
        "Validation": len(index[1]),
        "Classes": train.loc[index[1], "y"].nunique(),
    }
    for fold, index in cv.items()
])

assert cv_summary["Classes"].eq(4).all()
assert cv_summary["Validation"].sum() == len(train)

display(cv_summary)

,Fold,Training,Validation,Classes
0,1,6095,1524,4
1,2,6095,1524,4
2,3,6095,1524,4
3,4,6095,1524,4
4,5,6096,1523,4


## Part 3: Final training-sample tuning

Every registered configuration is evaluated on the fixed five folds. Mean validation macro F1 determines the selected specification. Exact ties use lower fold SD, then the deterministic weighting priority and candidate ID.

In [9]:
# 1: Final tuning function

selection_path = stage_6_dir / "stage_6_final_selection_register.csv"


def tune_model(model):
    model_candidates = (
        candidates.loc[candidates["Model"].eq(model)]
        .sort_values("Candidate ID")
        .reset_index(drop=True)
    )

    rows = []

    for weighting in weighting_modes(model):
        print(f"{model}: {weighting}")

        for _, row in model_candidates.iterrows():
            candidate_id = int(row["Candidate ID"])
            params = json.loads(row["Parameters"])
            scores = []
            warning_count = 0

            for fit_idx, val_idx in cv.values():
                pipe = make_pipeline(model, params, weighting)

                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    fit_pipeline(
                        pipe,
                        train.loc[fit_idx, predictors],
                        train.loc[fit_idx, "y"],
                        model,
                        weighting,
                    )

                warning_count += len(caught)
                pred = pipe.predict(train.loc[val_idx, predictors])
                scores.append(
                    f1_score(
                        train.loc[val_idx, "y"], pred,
                        labels=classes, average="macro", zero_division=0
                    )
                )

            rows.append({
                "Model": model,
                "Weighting mode": weighting,
                "Candidate ID": candidate_id,
                "Parameters": json.dumps(params, sort_keys=True),
                **{f"Fold {i} macro F1": score for i, score in enumerate(scores, 1)},
                "CV macro F1 mean": np.mean(scores),
                "CV macro F1 SD": np.std(scores, ddof=1),
                "Warnings": warning_count,
            })

    results = pd.DataFrame(rows)
    results["_weighting"] = results["Weighting mode"].map(weighting_priority)

    best = results.sort_values(
        ["CV macro F1 mean", "CV macro F1 SD", "_weighting", "Candidate ID"],
        ascending=[False, True, True, True],
    ).iloc[0]

    results = results.drop(columns="_weighting")
    results.to_csv(stage_6_dir / f"stage_6_final_tuning_{file_stub[model]}.csv", index=False)

    selected = pd.DataFrame([{
        "Model": model,
        "Selected weighting mode": best["Weighting mode"],
        "Selected candidate ID": int(best["Candidate ID"]),
        "Selected parameters": best["Parameters"],
        "CV macro F1 mean": float(best["CV macro F1 mean"]),
        "CV macro F1 SD": float(best["CV macro F1 SD"]),
    }])

    if selection_path.is_file():
        register = pd.read_csv(
            selection_path,
            keep_default_na=False,
            na_values=[""],
        )
        register = register.loc[~register["Model"].eq(model)]
        register = pd.concat([register, selected], ignore_index=True)
    else:
        register = selected

    register["_order"] = register["Model"].map({m: i for i, m in enumerate(model_order)})
    register.sort_values("_order").drop(columns="_order").to_csv(selection_path, index=False)

    display(selected)
    return results

In [10]:
# 2: MLR final tuning

mlr_search = tune_model("MLR")

MLR: None
MLR: Balanced


,Model,Selected weighting mode,Selected candidate ID,Selected parameters,CV macro F1 mean,CV macro F1 SD,_order
0,MLR,Balanced,1,"{""C"": 0.01}",0.394759,0.010135,0


In [11]:
# 3: RF final tuning

rf_search = tune_model("RF")

RF: None
RF: Balanced


,Model,Selected weighting mode,Selected candidate ID,Selected parameters,CV macro F1 mean,CV macro F1 SD
0,RF,Balanced,16,"{""max_depth"": 24, ""max_features"": ""sqrt"", ""min...",0.450227,0.018855


In [12]:
# 4: XGBoost final tuning

xgb_search = tune_model("XGBoost")

XGBoost: None
XGBoost: Balanced


,Model,Selected weighting mode,Selected candidate ID,Selected parameters,CV macro F1 mean,CV macro F1 SD
0,XGBoost,Balanced,9,"{""colsample_bytree"": 0.6, ""gamma"": 1.0, ""learn...",0.442507,0.01482


In [13]:
# 5: BRF final tuning

brf_search = tune_model("BRF")

BRF: Internal balanced sampling


,Model,Selected weighting mode,Selected candidate ID,Selected parameters,CV macro F1 mean,CV macro F1 SD
0,BRF,Internal balanced sampling,13,"{""max_depth"": 32, ""max_features"": ""log2"", ""min...",0.452768,0.021153


## Part 4: Final model fitting

The four selected specifications are fixed before test evaluation. Each model is fitted once using all 7,619 training participants. The majority-class dummy is fitted as the benchmark.

In [14]:
# 1: Check and save the four selected specifications

selected = pd.read_csv(
    selection_path,
    keep_default_na=False,
    na_values=[""],
)

expected_rows = {
    "MLR": 18,
    "RF": 120,
    "XGBoost": 120,
    "BRF": 60,
}
search_checks = []

for model in model_order:
    path = (
        stage_6_dir
        / f"stage_6_final_tuning_{file_stub[model]}.csv"
    )
    result = pd.read_csv(
        path,
        keep_default_na=False,
        na_values=[""],
    )
    search_checks.append(
        {
            "Model": model,
            "Expected rows": expected_rows[model],
            "Observed rows": len(result),
            "Five fold scores present": result[
                [f"Fold {i} macro F1" for i in range(1, 6)]
            ].notna().all().all(),
        }
    )

search_checks = pd.DataFrame(search_checks)
display(search_checks)

assert len(selected) == 4 and selected["Model"].is_unique
assert (
    search_checks["Expected rows"]
    == search_checks["Observed rows"]
).all()
assert search_checks["Five fold scores present"].all()

selected.to_csv(
    stage_6_dir / "stage_6_frozen_model_specifications.csv",
    index=False,
)
display(selected)


,Model,Expected rows,Observed rows,Five fold scores present
0,MLR,18,18,True
1,RF,120,120,True
2,XGBoost,120,120,True
3,BRF,60,60,True


,Model,Selected weighting mode,Selected candidate ID,Selected parameters,CV macro F1 mean,CV macro F1 SD
0,MLR,Balanced,1,"{""C"": 0.01}",0.394759,0.010135
1,RF,Balanced,16,"{""max_depth"": 24, ""max_features"": ""sqrt"", ""min...",0.450227,0.018855
2,XGBoost,Balanced,9,"{""colsample_bytree"": 0.6, ""gamma"": 1.0, ""learn...",0.442507,0.014820
3,BRF,Internal balanced sampling,13,"{""max_depth"": 32, ""max_features"": ""log2"", ""min...",0.452768,0.021153


In [15]:
# 2: Fit the four models on all training participants

models = {}
fit_rows = []

for model in model_order:
    spec = selected.loc[selected["Model"].eq(model)].iloc[0]
    weighting = spec["Selected weighting mode"]
    params = json.loads(spec["Selected parameters"])

    pipe = make_pipeline(model, params, weighting)

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        fit_pipeline(pipe, train[predictors], train["y"], model, weighting)

    models[model] = pipe
    fit_rows.append({
        "Model": model,
        "Weighting mode": weighting,
        "Candidate ID": int(spec["Selected candidate ID"]),
        "Warnings": len(caught),
    })

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(train[predictors], train["y"])
models["Dummy"] = dummy

fit_log = pd.DataFrame(fit_rows)
fit_log.to_csv(stage_6_dir / "stage_6_final_fit_log.csv", index=False)
display(fit_log)

,Model,Weighting mode,Candidate ID,Warnings
0,MLR,Balanced,1,0
1,RF,Balanced,16,0
2,XGBoost,Balanced,9,0
3,BRF,Internal balanced sampling,13,0


## Part 5: Held-out test evaluation

The held-out sample is loaded only after model selection and full-training fitting are complete. Its predictors and outcomes are then used for final evaluation only.


In [16]:
# 1: Load and check the held-out test sample

data = pd.read_csv(paths["data"], dtype={"NSID": "string"})
test = data.loc[data["NSID"].isin(test_ids)].copy().reset_index(drop=True)
del data

for column in numeric:
    test[column] = pd.to_numeric(test[column], errors="raise")

test["y"] = test["age18_outcome_code"].map(code_to_model).astype(int)

test_checks = {
    "Test sample contains 1,905 participants": len(test) == 1_905,
    "Test identifiers are unique": test["NSID"].is_unique,
    "Test identifiers match the reserved sample": set(test["NSID"]) == test_ids,
    "Training and test identifiers do not overlap": set(test["NSID"]).isdisjoint(train_ids),
    "All 69 predictors are present": set(predictors).issubset(test.columns),
    "All four classes are present": set(test["y"]) == set(classes),
}

test_audit = pd.DataFrame({"Check": test_checks.keys(), "Passed": test_checks.values()})
display(test_audit)

if not test_audit["Passed"].all():
    raise AssertionError("; ".join(test_audit.loc[~test_audit["Passed"], "Check"]))

,Check,Passed
0,"Test sample contains 1,905 participants",True
1,Test identifiers are unique,True
2,Test identifiers match the reserved sample,True
3,Training and test identifiers do not overlap,True
4,All 69 predictors are present,True
5,All four classes are present,True


In [17]:
# 2: Generate one prediction and probability vector per model and participant

evaluation_models = ["Dummy", "MLR", "RF", "XGBoost", "BRF"]
prediction_tables = []

for model in evaluation_models:
    fitted = models[model]

    # Generate the predicted destination and class probabilities
    # for each held-out participant.
    pred = fitted.predict(test[predictors]).astype(int)
    proba = fitted.predict_proba(test[predictors])

    estimator = fitted if model == "Dummy" else fitted.named_steps["model"]

    # Align probability columns to the fixed four destination classes
    # so all models use the same outcome order.
    aligned = np.zeros((len(test), 4))

    for j, class_code in enumerate(estimator.classes_.astype(int)):
        aligned[:, class_code] = proba[:, j]

    table = pd.DataFrame({
        "NSID": test["NSID"],
        "Model": model,
        "Actual internal class": test["y"],
        "Predicted internal class": pred,
        "Probability Education": aligned[:, 0],
        "Probability Employment": aligned[:, 1],
        "Probability Apprenticeship or training": aligned[:, 2],
        "Probability Unemployment or inactivity (NEET)": aligned[:, 3],
    })

    table["Actual outcome"] = table["Actual internal class"].map(model_to_label)
    table["Predicted outcome"] = table["Predicted internal class"].map(model_to_label)

    prediction_tables.append(table)

test_predictions = pd.concat(prediction_tables, ignore_index=True)

# Confirm that the four class probabilities form a valid probability distribution.
probability_columns = [
    column
    for column in test_predictions.columns
    if column.startswith("Probability ")
]

assert np.allclose(
    test_predictions[probability_columns].sum(axis=1),
    1.0
)

# Save participant-level predictions for evaluation and bootstrap analysis.
test_predictions.to_csv(
    stage_6_dir / "stage_6_test_predictions.csv",
    index=False
)

print(f"Prediction rows: {len(test_predictions):,}")

Prediction rows: 9,525


In [18]:
# 3: Overall and class-specific performance

overall_rows = []
class_rows = []

for model in evaluation_models:
    table = test_predictions.loc[test_predictions["Model"].eq(model)]
    y_true = table["Actual internal class"].to_numpy()
    y_pred = table["Predicted internal class"].to_numpy()

    # Calculate overall test performance for each model.
    overall_rows.append({"Model": model, **overall_metrics(y_true, y_pred)})

    # Calculate precision, recall and F1 separately for each destination.
    detail = class_metrics(y_true, y_pred)
    detail.insert(0, "Model", model)
    class_rows.append(detail)

test_overall = pd.DataFrame(overall_rows)
test_class = pd.concat(class_rows, ignore_index=True)

test_overall.to_csv(stage_6_dir / "stage_6_test_overall_performance.csv", index=False)
test_class.to_csv(stage_6_dir / "stage_6_test_class_performance.csv", index=False)

display(test_overall.round(4))
display(test_class.round(4))

,Model,Macro F1,Balanced accuracy,MCC,Accuracy,Weighted F1
0,Dummy,0.1711,0.2500,0.0000,0.5202,0.3560
1,MLR,0.3905,0.4317,0.2588,0.5024,0.5196
2,RF,0.4378,0.4284,0.3294,0.5948,0.5838
3,XGBoost,0.4157,0.4141,0.2909,0.5549,0.5576
4,BRF,0.4309,0.4385,0.2964,0.5559,0.5589


,Model,Internal class,Outcome code,Outcome,Precision,Recall,F1,Support
0,Dummy,0,1,Education,0.5202,1.0000,0.6844,991
1,Dummy,1,4,Employment,0.0000,0.0000,0.0000,560
2,Dummy,2,5,Apprenticeship or training,0.0000,0.0000,0.0000,104
3,Dummy,3,6,Unemployment or inactivity (NEET),0.0000,0.0000,0.0000,250
4,MLR,0,1,Education,0.7400,0.6720,0.7044,991
5,MLR,1,4,Employment,0.4399,0.2875,0.3477,560
6,MLR,2,5,Apprenticeship or training,0.1384,0.4231,0.2085,104
7,MLR,3,6,Unemployment or inactivity (NEET),0.2679,0.3440,0.3012,250
8,RF,0,1,Education,0.7185,0.7780,0.7471,991
9,RF,1,4,Employment,0.4772,0.4857,0.4814,560


In [19]:
# 4: Confusion matrices and observed/predicted distributions

confusion_rows = []
distribution_rows = []

for model in evaluation_models:
    table = test_predictions.loc[test_predictions["Model"].eq(model)]
    y_true = table["Actual internal class"].to_numpy()
    y_pred = table["Predicted internal class"].to_numpy()

    # Create raw and row-normalised confusion matrices for each model.
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    row_prop = cm / cm.sum(axis=1, keepdims=True)

    for i, actual in enumerate(classes):
        for j, predicted in enumerate(classes):
            confusion_rows.append({
                "Model": model,
                "Actual outcome": model_to_label[actual],
                "Predicted outcome": model_to_label[predicted],
                "Count": int(cm[i, j]),
                "Row proportion": float(row_prop[i, j]),
            })

    # Compare the observed and predicted class distributions.
    for kind, values in [("Observed", y_true), ("Predicted", y_pred)]:
        counts = pd.Series(values).value_counts().reindex(classes, fill_value=0)

        for class_code in classes:
            distribution_rows.append({
                "Model": model,
                "Distribution": kind,
                "Outcome": model_to_label[class_code],
                "Participants": int(counts[class_code]),
                "Proportion": counts[class_code] / len(values),
            })

confusion_table = pd.DataFrame(confusion_rows)
distribution_table = pd.DataFrame(distribution_rows)

confusion_table.to_csv(
    stage_6_dir / "stage_6_test_confusion_matrix.csv",
    index=False
)

distribution_table.to_csv(
    stage_6_dir / "stage_6_test_observed_predicted_distributions.csv",
    index=False
)

display(distribution_table)

,Model,Distribution,Outcome,Participants,Proportion
0,Dummy,Observed,Education,991,0.520210
1,Dummy,Observed,Employment,560,0.293963
2,Dummy,Observed,Apprenticeship or training,104,0.054593
3,Dummy,Observed,Unemployment or inactivity (NEET),250,0.131234
4,Dummy,Predicted,Education,1905,1.000000
5,Dummy,Predicted,Employment,0,0.000000
6,Dummy,Predicted,Apprenticeship or training,0,0.000000
7,Dummy,Predicted,Unemployment or inactivity (NEET),0,0.000000
8,MLR,Observed,Education,991,0.520210
9,MLR,Observed,Employment,560,0.293963


## Part 6: Paired bootstrap uncertainty

Uncertainty is estimated from 5,000 stratified paired bootstrap resamples of the held-out predictions. Each destination is resampled separately so its observed test count is preserved, and the same sampled participant indices are used for every model. The classifiers are not refitted during bootstrap resampling.


In [20]:
# 1: Align predictions for paired bootstrap resampling

BOOTSTRAP_REPEATS = 5_000
BOOTSTRAP_RANDOM_STATE = 424242
bootstrap_models = ["MLR", "RF", "XGBoost", "BRF"]

reference = (
    test_predictions.loc[
        test_predictions["Model"].eq("MLR"),
        ["NSID", "Actual internal class"],
    ]
    .sort_values("NSID")
    .reset_index(drop=True)
)

ids = reference["NSID"].tolist()
y_true = reference["Actual internal class"].to_numpy(
    dtype=int
)

predictions = {
    model: (
        test_predictions.loc[
            test_predictions["Model"].eq(model)
        ]
        .set_index("NSID")
        .loc[ids, "Predicted internal class"]
        .to_numpy(dtype=int)
    )
    for model in bootstrap_models
}

class_positions = {
    class_code: np.flatnonzero(y_true == class_code)
    for class_code in classes
}

print(
    {
        model_to_label[class_code]: len(positions)
        for class_code, positions in class_positions.items()
    }
)


{'Education': 991, 'Employment': 560, 'Apprenticeship or training': 104, 'Unemployment or inactivity (NEET)': 250}


In [21]:
# 2: Run the stratified paired bootstrap

rng = np.random.default_rng(BOOTSTRAP_RANDOM_STATE)
overall_boot_rows = []
class_boot_rows = []

for bootstrap_id in range(1, BOOTSTRAP_REPEATS + 1):
    sample = np.concatenate(
        [
            rng.choice(
                positions,
                size=len(positions),
                replace=True,
            )
            for positions in class_positions.values()
        ]
    )
    y_bootstrap = y_true[sample]

    for model in bootstrap_models:
        pred_bootstrap = predictions[model][sample]

        # Metrics are recalculated from fixed held-out predictions;
        # no model is refitted inside the bootstrap.
        overall_boot_rows.append(
            {
                "Bootstrap": bootstrap_id,
                "Model": model,
                **overall_metrics(
                    y_bootstrap,
                    pred_bootstrap,
                ),
            }
        )

        detail = class_metrics(
            y_bootstrap,
            pred_bootstrap,
        )
        detail.insert(0, "Model", model)
        detail.insert(0, "Bootstrap", bootstrap_id)
        class_boot_rows.append(detail)

bootstrap_overall = pd.DataFrame(overall_boot_rows)
bootstrap_class = pd.concat(
    class_boot_rows,
    ignore_index=True,
)

bootstrap_overall.to_csv(
    stage_6_dir
    / "stage_6_bootstrap_overall_resamples.csv",
    index=False,
)
bootstrap_class.to_csv(
    stage_6_dir
    / "stage_6_bootstrap_class_resamples.csv",
    index=False,
)

print(
    f"Overall bootstrap rows: {len(bootstrap_overall):,}"
)
print(
    f"Class bootstrap rows: {len(bootstrap_class):,}"
)


Overall bootstrap rows: 20,000
Class bootstrap rows: 80,000


In [22]:
# 3: Bootstrap intervals and paired model differences

def interval(values):
    return np.quantile(values, [0.025, 0.975])


overall_intervals = []
for model in bootstrap_models:
    point = test_overall.loc[test_overall["Model"].eq(model)].iloc[0]
    boot = bootstrap_overall.loc[bootstrap_overall["Model"].eq(model)]

    for metric in ["Macro F1", "Balanced accuracy", "MCC", "Accuracy", "Weighted F1"]:
        low, high = interval(boot[metric])
        overall_intervals.append({
            "Model": model, "Metric": metric, "Point estimate": point[metric],
            "Lower 95%": low, "Upper 95%": high,
        })

class_intervals = []
for (model, class_code), boot in bootstrap_class.groupby(["Model", "Internal class"]):
    point = test_class.loc[
        test_class["Model"].eq(model) & test_class["Internal class"].eq(class_code)
    ].iloc[0]

    for metric in ["Precision", "Recall", "F1"]:
        low, high = interval(boot[metric])
        class_intervals.append({
            "Model": model, "Outcome": model_to_label[class_code], "Metric": metric,
            "Point estimate": point[metric], "Lower 95%": low, "Upper 95%": high,
        })

comparisons = [("RF", "MLR"), ("XGBoost", "MLR"), ("BRF", "RF"), ("RF", "XGBoost")]
difference_rows = []
difference_intervals = []

for first, second in comparisons:
    first_boot = bootstrap_overall.loc[bootstrap_overall["Model"].eq(first)].set_index("Bootstrap")
    second_boot = bootstrap_overall.loc[bootstrap_overall["Model"].eq(second)].set_index("Bootstrap")

    for metric in ["Macro F1", "Balanced accuracy"]:
        diff = first_boot[metric] - second_boot[metric]
        low, high = interval(diff)

        first_point = test_overall.loc[test_overall["Model"].eq(first), metric].iloc[0]
        second_point = test_overall.loc[test_overall["Model"].eq(second), metric].iloc[0]

        difference_intervals.append({
            "Comparison": f"{first} - {second}", "Metric": metric,
            "Point difference": first_point - second_point,
            "Lower 95%": low, "Upper 95%": high,
        })

        difference_rows.append(pd.DataFrame({
            "Bootstrap": diff.index,
            "Comparison": f"{first} - {second}",
            "Metric": metric,
            "Difference": diff.values,
        }))

overall_intervals = pd.DataFrame(overall_intervals)
class_intervals = pd.DataFrame(class_intervals)
difference_resamples = pd.concat(difference_rows, ignore_index=True)
difference_intervals = pd.DataFrame(difference_intervals)

overall_intervals.to_csv(stage_6_dir / "stage_6_bootstrap_overall_intervals.csv", index=False)
class_intervals.to_csv(stage_6_dir / "stage_6_bootstrap_class_intervals.csv", index=False)
difference_resamples.to_csv(stage_6_dir / "stage_6_bootstrap_model_difference_resamples.csv", index=False)
difference_intervals.to_csv(stage_6_dir / "stage_6_bootstrap_model_difference_intervals.csv", index=False)

display(overall_intervals.round(4))
display(difference_intervals.round(4))

,Model,Metric,Point estimate,Lower 95%,Upper 95%
0,MLR,Macro F1,0.3905,0.3687,0.4123
1,MLR,Balanced accuracy,0.4317,0.4010,0.4618
2,MLR,MCC,0.2588,0.2294,0.2882
3,MLR,Accuracy,0.5024,0.4814,0.5234
4,MLR,Weighted F1,0.5196,0.4992,0.5394
5,RF,Macro F1,0.4378,0.4104,0.4663
6,RF,Balanced accuracy,0.4284,0.4046,0.4540
7,RF,MCC,0.3294,0.2973,0.3628
8,RF,Accuracy,0.5948,0.5753,0.6147
9,RF,Weighted F1,0.5838,0.5641,0.6040


,Comparison,Metric,Point difference,Lower 95%,Upper 95%
0,RF - MLR,Macro F1,0.0473,0.0218,0.0721
1,RF - MLR,Balanced accuracy,-0.0033,-0.0308,0.0240
2,XGBoost - MLR,Macro F1,0.0253,0.0023,0.0480
3,XGBoost - MLR,Balanced accuracy,-0.0176,-0.0435,0.0078
4,BRF - RF,Macro F1,-0.0068,-0.0261,0.0128
5,BRF - RF,Balanced accuracy,0.0101,-0.0085,0.0289
6,RF - XGBoost,Macro F1,0.0220,0.0021,0.0426
7,RF - XGBoost,Balanced accuracy,0.0143,-0.0044,0.0334


## Part 7: Completion audit


In [23]:
# 1: Final checks

final_checks = {
    "Four model specifications are present": (
        len(selected) == 4
        and selected["Model"].is_unique
    ),
    "All four final fits completed": (
        set(fit_log["Model"]) == set(model_order)
    ),
    "Five evaluation models are present": (
        set(test_predictions["Model"])
        == set(evaluation_models)
    ),
    "Each model has 1,905 predictions": (
        test_predictions.groupby("Model")
        .size()
        .eq(1_905)
        .all()
    ),
    "Each model has 1,905 unique identifiers": (
        test_predictions.groupby("Model")["NSID"]
        .nunique()
        .eq(1_905)
        .all()
    ),
    "Probability rows sum to one": np.allclose(
        test_predictions[probability_columns].sum(axis=1),
        1.0,
    ),
    "Overall results contain five models": (
        set(test_overall["Model"])
        == set(evaluation_models)
    ),
    "Class results contain four destinations per model": (
        test_class.groupby("Model")["Internal class"]
        .nunique()
        .eq(4)
        .all()
    ),
    "Stratified bootstrap has 5,000 replicates per model": (
        bootstrap_overall.groupby("Model")["Bootstrap"]
        .nunique()
        .eq(5_000)
        .all()
    ),
}

final_audit = pd.DataFrame(
    {
        "Check": final_checks.keys(),
        "Passed": final_checks.values(),
    }
)
final_audit.to_csv(
    stage_6_dir / "stage_6_final_audit.csv",
    index=False,
)
display(final_audit)

if not final_audit["Passed"].all():
    failed = final_audit.loc[
        ~final_audit["Passed"],
        "Check",
    ].tolist()
    raise AssertionError("; ".join(failed))


,Check,Passed
0,Four model specifications are present,True
1,All four final fits completed,True
2,Five evaluation models are present,True
3,"Each model has 1,905 predictions",True
4,"Each model has 1,905 unique identifiers",True
5,Probability rows sum to one,True
6,Overall results contain five models,True
7,Class results contain four destinations per model,True
8,"Stratified bootstrap has 5,000 replicates per ...",True


In [24]:
# 2: Output manifest and completion record

expected_files = [
    "stage_6_final_tuning_mlr.csv",
    "stage_6_final_tuning_rf.csv",
    "stage_6_final_tuning_xgboost.csv",
    "stage_6_final_tuning_brf.csv",
    "stage_6_final_selection_register.csv",
    "stage_6_frozen_model_specifications.csv",
    "stage_6_final_fit_log.csv",
    "stage_6_test_predictions.csv",
    "stage_6_test_overall_performance.csv",
    "stage_6_test_class_performance.csv",
    "stage_6_test_confusion_matrix.csv",
    "stage_6_test_observed_predicted_distributions.csv",
    "stage_6_bootstrap_overall_resamples.csv",
    "stage_6_bootstrap_class_resamples.csv",
    "stage_6_bootstrap_overall_intervals.csv",
    "stage_6_bootstrap_class_intervals.csv",
    "stage_6_bootstrap_model_difference_resamples.csv",
    "stage_6_bootstrap_model_difference_intervals.csv",
    "stage_6_final_audit.csv",
]

manifest = pd.DataFrame(
    {
        "File": expected_files,
        "Exists": [
            (stage_6_dir / name).is_file()
            for name in expected_files
        ],
    }
)

if not manifest["Exists"].all():
    missing_outputs = manifest.loc[
        ~manifest["Exists"],
        "File",
    ].tolist()
    raise FileNotFoundError(
        "Missing output file(s): "
        + ", ".join(missing_outputs)
    )

manifest.to_csv(
    stage_6_dir / "stage_6_output_manifest.csv",
    index=False,
)

completion = pd.DataFrame(
    [
        {
            "Required checks passed": bool(
                final_audit["Passed"].all()
            ),
            "Required checks": len(final_audit),
            "Failed required checks": int(
                (~final_audit["Passed"]).sum()
            ),
            "Held-out participants": len(test),
            "Bootstrap repeats": BOOTSTRAP_REPEATS,
            "Output files present": bool(
                manifest["Exists"].all()
            ),
        }
    ]
)

completion.to_csv(
    stage_6_dir / "stage_6_completion.csv",
    index=False,
)

display(manifest)
display(completion)
print("Stage 6 complete.")


,File,Exists
0,stage_6_final_tuning_mlr.csv,True
1,stage_6_final_tuning_rf.csv,True
2,stage_6_final_tuning_xgboost.csv,True
3,stage_6_final_tuning_brf.csv,True
4,stage_6_final_selection_register.csv,True
5,stage_6_frozen_model_specifications.csv,True
6,stage_6_final_fit_log.csv,True
7,stage_6_test_predictions.csv,True
8,stage_6_test_overall_performance.csv,True
9,stage_6_test_class_performance.csv,True


,Required checks passed,Required checks,Failed required checks,Held-out participants,Bootstrap repeats,Output files present
0,True,9,0,1905,5000,True


Stage 6 complete.


## Stage 6 summary

Stage 6 fixes one specification for each model using five-fold cross-validation on the training sample, refits the selected specifications on all 7,619 training participants, and evaluates them once on the 1,905-participant held-out test sample.

Held-out performance is recorded using overall and class-specific metrics, confusion matrices and observed/predicted class distributions. Uncertainty and paired model differences are estimated from 5,000 stratified paired bootstrap resamples of the saved test predictions.
